# 🎙️ Telugu ASR Pipeline — File 2 of 3
## Model Prototyping & Pipeline Validation

---

### 🎯 Objective
Validate the **end-to-end Wav2Vec2 CTC training pipeline** on a 500-sample Telugu subset before committing to a full fine-tuning run in a production job. Every stage — tokenizer, feature extractor, preprocessing, forward pass, and gradient update — is exercised here so failures are caught cheaply.

---

### 📋 Prerequisites (outputs from File 1)

| Artifact | Path | Description |
|---|---|---|
| Clean HF Dataset | `./telugu_asr_clean_dataset/` | Arrow format, columns: `audio`, `transcription`, `audio_id` |
| Character Inventory | `./character_inventory.json` | `{num_samples, num_unique_chars, characters: [...]}` |

---

### 🗺️ Notebook Flow
```
Install → GPU Check → Load Dataset (500-sample subset)
   → Build vocab.json → Init Processor → Preprocess Subset
   → Load Wav2Vec2 Model → Dummy Forward Pass
   → Dummy Training Loop (3–5 steps) → Summary
```

---

### ➡️ Downstream
| File | Purpose |
|---|---|
| `03_evaluation_and_inference.ipynb` | Full WER/CER evaluation + inference demos on the trained checkpoint |

## 📦 Section 1 — Environment Setup

**Rationale:** Pin `datasets<3.0` to avoid the `torchcodec` audio backend introduced in 3.x (which is unavailable in this environment). All other packages are unpinned — the GPU server's conda base already has compatible versions.

In [1]:
# ═══════════════════════════════════════════════════════════════
# SECTION 1 — ENVIRONMENT SETUP
# ═══════════════════════════════════════════════════════════════

# - datasets pinned to 2.x: datasets>=3.0 switched to torchcodec which requires
#   FFmpeg shared libs (libavutil.so.*) not present on this machine
# - torchaudio excluded: compiled against CUDA 13 (libcudart.so.13), not present here
# - ipywidgets: required for tqdm progress bars inside Jupyter
%pip install -q "datasets>=2.14,<3.0" transformers torch accelerate soundfile librosa \
               evaluate jiwer huggingface_hub ipywidgets

# NOTE: If this is the first run, restart the kernel after this cell, then
# re-run from here so the freshly-installed datasets 2.x version is active.

Note: you may need to restart the kernel to use updated packages.


In [1]:
# Verify datasets 2.x is active in this kernel — if this fails, restart the kernel
import datasets as _ds, importlib
importlib.reload(_ds)
assert int(_ds.__version__.split(".")[0]) < 3, (
    f"datasets {_ds.__version__} detected — restart the kernel so the 2.x install takes effect."
)
print(f"✅ datasets version : {_ds.__version__}  (2.x — soundfile backend active)")

✅ datasets version : 2.21.0  (2.x — soundfile backend active)


In [2]:
# ───────────────────────────────────────────────────────────────
# SECTION 1 — IMPORTS
# ───────────────────────────────────────────────────────────────

import os
import json
import warnings

import torch
import numpy as np
from datasets import load_from_disk, Audio   # Audio needed for cast_column soundfile fix

from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print("✅ All imports successful.")

✅ All imports successful.


## 🖥️ Section 2 — GPU Availability Check

**Rationale:** Fail fast if the GPU is not visible. The A6000 has 49 GB VRAM — we print device name and free memory so we know how much headroom we have before loading the 300M-parameter model.

In [3]:
# ═══════════════════════════════════════════════════════════════
# SECTION 2 — GPU AVAILABILITY CHECK
# ═══════════════════════════════════════════════════════════════

# Assert CUDA is available — fail immediately rather than silently running on CPU
assert torch.cuda.is_available(), "❌ CUDA not available. Check driver and CUDA installation."

DEVICE: torch.device = torch.device("cuda")  # Primary device for all tensors and models

gpu_device_name: str = torch.cuda.get_device_name(0)                           # Human-readable GPU name
gpu_total_vram_gb: float = torch.cuda.get_device_properties(0).total_memory / 1e9  # Total VRAM in GB
gpu_free_vram_gb: float = (                                                     # Free VRAM = total - reserved
    torch.cuda.get_device_properties(0).total_memory
    - torch.cuda.memory_reserved(0)
) / 1e9

print("🖥️  GPU Device Info:")
print(f"   Device name   : {gpu_device_name}")            # E.g. 'NVIDIA RTX A6000'
print(f"   Total VRAM    : {gpu_total_vram_gb:.1f} GB")   # E.g. '49.0 GB'
print(f"   Free VRAM     : {gpu_free_vram_gb:.1f} GB")    # Available before model load
print(f"   PyTorch ver.  : {torch.__version__}")          # Torch version for reproducibility
print(f"   CUDA ver.     : {torch.version.cuda}")         # CUDA runtime version
print("✅ GPU ready.")  # Confirm GPU check passed

🖥️  GPU Device Info:
   Device name   : NVIDIA RTX A6000
   Total VRAM    : 51.0 GB
   Free VRAM     : 51.0 GB
   PyTorch ver.  : 2.6.0+cu124
   CUDA ver.     : 12.4
✅ GPU ready.


## 📂 Section 3 — Load Dataset & Extract 500-Sample Subset

**Rationale:** We prototype on 500 samples rather than the full ~44K to keep iteration time under a minute. The subset is drawn deterministically (fixed seed) so results are reproducible across runs.

In [4]:
# ═══════════════════════════════════════════════════════════════
# SECTION 3 — LOAD DATASET & EXTRACT 500-SAMPLE SUBSET
# ═══════════════════════════════════════════════════════════════

CLEAN_DATASET_PATH: str = "./telugu_asr_clean_dataset"
SUBSET_SIZE: int = 500
SUBSET_SEED: int = 42

assert os.path.isdir(CLEAN_DATASET_PATH), (
    f"❌ Dataset not found at '{CLEAN_DATASET_PATH}'. Run File 1 first."
)

print(f"📂 Loading clean dataset from '{CLEAN_DATASET_PATH}'...")
full_dataset = load_from_disk(CLEAN_DATASET_PATH)

# Re-cast audio column — ensures datasets uses soundfile for decoding (torchaudio not required)
full_dataset = full_dataset.cast_column("audio", Audio(sampling_rate=16_000))

print(f"📊 Full dataset size     : {len(full_dataset)} samples")
print(f"📊 Dataset features      : {full_dataset.features}")

proto_dataset = full_dataset.shuffle(seed=SUBSET_SEED).select(range(SUBSET_SIZE))

print(f"\n✅ Prototype subset      : {len(proto_dataset)} samples (seed={SUBSET_SEED})")

# Column-wise access — avoids row[0] which decodes ALL columns including audio
print(f"📋 Sample transcription  : {full_dataset['transcription'][0]}")
print(f"📋 Sample audio_id       : {full_dataset['audio_id'][0]}")

📂 Loading clean dataset from './telugu_asr_clean_dataset'...
📊 Full dataset size     : 35426 samples
📊 Dataset features      : {'audio': Audio(sampling_rate=16000, mono=True, decode=True, id=None), 'transcription': Value(dtype='string', id=None), 'audio_id': Value(dtype='string', id=None)}

✅ Prototype subset      : 500 samples (seed=42)
📋 Sample transcription  : కచ్చితంగా చూపిస్తుంది కదా మరి
📋 Sample audio_id       : TE2406-TE2408_1-A.089.wav


## 🔤 Section 4 — Build `vocab.json`

**Rationale:** Wav2Vec2CTC requires a character-level vocabulary where every unique output token maps to a unique integer ID. We build this from the **full dataset** (not just the 500-sample subset) so the vocabulary is complete and no character is OOV during actual fine-tuning. Three special tokens are added: `|` (word boundary replacing space), `[UNK]` (unknown character), `[PAD]` (CTC blank/padding).

In [5]:
# ═══════════════════════════════════════════════════════════════
# SECTION 4 — BUILD vocab.json
# ═══════════════════════════════════════════════════════════════

VOCAB_SAVE_PATH: str = "./vocab.json"                 # Output path for the vocabulary file
CHARACTER_INVENTORY_PATH: str = "./character_inventory.json"  # Written by File 1

# ── Load the character list from File 1's character inventory ──
# character_inventory.json contains: {num_samples, num_unique_chars, characters: [...]}
# Using the pre-computed inventory avoids re-scanning 44K transcriptions here
assert os.path.isfile(CHARACTER_INVENTORY_PATH), (
    f"❌ '{CHARACTER_INVENTORY_PATH}' not found. Run File 1 first."
)

with open(CHARACTER_INVENTORY_PATH, "r", encoding="utf-8") as inv_file:  # Open inventory JSON
    inventory_data: dict = json.load(inv_file)  # Parse JSON into Python dict

raw_characters: list = inventory_data["characters"]  # Sorted list of all Telugu characters
print(f"📊 Characters from inventory : {len(raw_characters)} unique chars")  # Report count

# ── Build the vocabulary mapping: character → integer ID ──
# Enumerate starting from 0; special tokens are appended at the end
vocab_dict: dict = {char: idx for idx, char in enumerate(raw_characters)}  # Char-to-int map

# Add the word-boundary token "|" which replaces spaces in CTC decoding
# Wav2Vec2 treats "|" as the delimiter between words during greedy/beam decoding
vocab_dict["|"] = len(vocab_dict)   # Assign next available integer ID to "|"

# Add [UNK] token for any character seen at inference time but not in training vocab
vocab_dict["[UNK]"] = len(vocab_dict)  # Assign next available integer ID to [UNK]

# Add [PAD] token — used as the CTC blank token and for sequence-length padding
# IMPORTANT: [PAD] must NOT be 0 if 0 is already assigned to a real character
# We append it last so its ID is always num_chars + 2
vocab_dict["[PAD]"] = len(vocab_dict)  # Assign next available integer ID to [PAD]

# Write the vocabulary dictionary to vocab.json with UTF-8 encoding
# ensure_ascii=False is required to preserve Telugu Unicode characters as-is
with open(VOCAB_SAVE_PATH, "w", encoding="utf-8") as vocab_file:  # Open for writing
    json.dump(vocab_dict, vocab_file, ensure_ascii=False, indent=2)  # Pretty-print JSON

print(f"\n📊 Vocabulary Summary:")
print(f"   Telugu characters  : {len(raw_characters)}")          # Base chars from inventory
print(f"   + '|' (word sep)   : ID {vocab_dict['|']}")           # Word boundary token ID
print(f"   + '[UNK]'          : ID {vocab_dict['[UNK]']}")       # Unknown token ID
print(f"   + '[PAD]'          : ID {vocab_dict['[PAD]']}")       # Padding/blank token ID
print(f"   Total vocab size   : {len(vocab_dict)}")              # Total vocabulary size
print(f"\n✅ vocab.json saved to '{VOCAB_SAVE_PATH}'")  # Confirm save

📊 Characters from inventory : 64 unique chars

📊 Vocabulary Summary:
   Telugu characters  : 64
   + '|' (word sep)   : ID 64
   + '[UNK]'          : ID 65
   + '[PAD]'          : ID 66
   Total vocab size   : 67

✅ vocab.json saved to './vocab.json'


## ⚙️ Section 5 — Initialize Processor

**Rationale:** `Wav2Vec2Processor` bundles the tokenizer and feature extractor into a single API. The tokenizer converts Telugu text ↔ token IDs. The feature extractor normalizes raw 16 kHz waveforms to zero-mean unit-variance `input_values`. Saving the processor means File 3 can load it without rebuilding the vocab.

In [6]:
# ═══════════════════════════════════════════════════════════════
# SECTION 5 — INITIALIZE WAV2VEC2 PROCESSOR
# ═══════════════════════════════════════════════════════════════

PROCESSOR_SAVE_PATH: str = "./telugu_wav2vec2_processor"  # Directory to save processor files

# ── Initialize the CTC Tokenizer from our custom vocab.json ──
tokenizer = Wav2Vec2CTCTokenizer(
    VOCAB_SAVE_PATH,          # Path to the vocab.json we just built
    unk_token="[UNK]",        # Token emitted for characters not in vocab
    pad_token="[PAD]",        # Token used for CTC blank and sequence padding
    word_delimiter_token="|", # Token that marks word boundaries (replaces spaces)
)
print(f"✅ Tokenizer initialized  | vocab size: {tokenizer.vocab_size}")  # Confirm tokenizer

# ── Initialize the Feature Extractor ──
# The feature extractor converts raw numpy waveforms → normalized float32 input_values
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,           # Mono audio: 1 channel
    sampling_rate=16000,      # Expected input sample rate (matches File 1's 16 kHz WAVs)
    padding_value=0.0,        # Value used to pad waveforms to equal length in a batch
    do_normalize=True,        # Normalize each waveform to zero mean, unit variance
    return_attention_mask=True,  # Return an attention mask so the model ignores padding
)
print(f"✅ Feature extractor initialized | sampling_rate: {feature_extractor.sampling_rate} Hz")  # Confirm

# ── Combine into a single Wav2Vec2Processor ──
# Processor exposes processor(audio) for feature extraction and processor.tokenizer for text
processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,  # Audio preprocessing component
    tokenizer=tokenizer,                  # Text tokenization component
)

# Save processor to disk so File 3 can load it with Wav2Vec2Processor.from_pretrained()
processor.save_pretrained(PROCESSOR_SAVE_PATH)  # Writes tokenizer_config.json, vocab.json, etc.
print(f"✅ Processor saved to '{PROCESSOR_SAVE_PATH}'")  # Confirm save

# ── Sanity test: encode a Telugu sentence then decode it back ──
SANITY_TEST_TEXT: str = "నమస్కారం తెలుగు"  # Sample Telugu sentence for encode/decode test

# Encode: text → token IDs (space is automatically replaced by "|")
encoded_ids: list = tokenizer(SANITY_TEST_TEXT).input_ids  # Tokenize to integer IDs

# Decode: token IDs → text (group_tokens=False keeps all tokens including CTC duplicates)
decoded_text: str = tokenizer.decode(encoded_ids)  # Reverse-map IDs back to characters

print(f"\n🧪 Tokenizer Sanity Test:")
print(f"   Input text     : {repr(SANITY_TEST_TEXT)}")  # Original text
print(f"   Encoded IDs    : {encoded_ids}")              # Integer token IDs
print(f"   Decoded text   : {repr(decoded_text)}")       # Reconstructed text

# Assert round-trip is lossless: decoded text should match input (spaces → '|' → spaces)
# The decoder maps '|' back to space, so we compare after normalizing spaces
decoded_normalized: str = decoded_text.replace("|", " ").strip()  # Replace '|' back to space
assert decoded_normalized == SANITY_TEST_TEXT.strip(), (
    f"❌ Round-trip mismatch!\n  Expected: {repr(SANITY_TEST_TEXT)}\n  Got: {repr(decoded_normalized)}"
)
print("✅ Sanity test PASSED — tokenizer encode/decode is lossless.")  # Confirm round-trip

✅ Tokenizer initialized  | vocab size: 67
✅ Feature extractor initialized | sampling_rate: 16000 Hz
✅ Processor saved to './telugu_wav2vec2_processor'

🧪 Tokenizer Sanity Test:
   Input text     : 'నమస్కారం తెలుగు'
   Encoded IDs    : [34, 39, 47, 61, 15, 49, 41, 0, 64, 30, 55, 42, 52, 17, 52]
   Decoded text   : 'నమస్కారం తెలుగు'
✅ Sanity test PASSED — tokenizer encode/decode is lossless.


## 🔄 Section 6 — Preprocess the 500-Sample Subset

**Rationale:** `prepare_dataset` converts each raw sample into the tensors the model expects: `input_values` (normalized waveform float32) and `labels` (token ID list). We attach `input_length` so the data collator can compute CTC input lengths correctly. The mapping is done over the 500-sample subset only — preprocessing all 44K samples would be done in File 3 with caching.

In [7]:
# ═══════════════════════════════════════════════════════════════
# SECTION 6 — PREPROCESS THE 500-SAMPLE SUBSET
# ═══════════════════════════════════════════════════════════════

def prepare_dataset(batch: dict) -> dict:
    """Convert one raw dataset sample into model-ready tensors.

    Steps:
    1. Extract the decoded audio array from the HuggingFace Audio feature dict.
    2. Run the feature extractor to produce normalized input_values.
    3. Encode the transcription text into a list of integer label IDs.
    4. Record input_length (number of waveform frames) for CTC length computation.

    Args:
        batch: A single sample dict with keys 'audio', 'transcription', 'audio_id'.

    Returns:
        The same dict with added keys: input_values, labels, input_length.
    """
    audio_data: dict = batch["audio"]  # HuggingFace Audio dict: {array, sampling_rate, path}
    waveform_array: np.ndarray = audio_data["array"]      # Raw float32 waveform as numpy array
    sample_rate: int = audio_data["sampling_rate"]        # Should be 16000 (set in File 1)

    # Feature extractor: normalize waveform → input_values (zero-mean, unit-variance)
    # sampling_rate is passed to let the extractor verify/resample if needed
    feature_extraction_output = processor(
        waveform_array,                # Raw numpy waveform
        sampling_rate=sample_rate,     # Tell extractor what the source sample rate is
        return_tensors="np",           # Return numpy arrays (not torch tensors) for .map() compat
    )
    # input_values shape: (1, num_frames) — squeeze the batch dim since this is one sample
    batch["input_values"] = feature_extraction_output.input_values[0]  # Shape: (num_frames,)

    # Tokenizer: encode transcription text → list of integer label IDs
    # with_special_tokens=False: CTC training labels should NOT include BOS/EOS tokens
    label_encoding = processor.tokenizer(
        batch["transcription"],        # Telugu transcription string
        return_tensors=None,           # Return plain Python list, not torch tensor
    )
    batch["labels"] = label_encoding.input_ids  # List of integer character IDs

    # Store input_length (waveform frames count) for CTC loss length calculation
    batch["input_length"] = len(batch["input_values"])  # Number of waveform time steps

    return batch  # Return the sample dict with the three new fields added

print("🔄 Mapping prepare_dataset over the 500-sample subset...")  # Progress message

# Apply prepare_dataset to every sample in the prototype subset
# remove_columns drops the original raw columns we no longer need after preprocessing
preprocessed_dataset = proto_dataset.map(
    prepare_dataset,                          # Function to apply per sample
    remove_columns=["audio", "audio_id"],     # Drop raw audio dict and id (not needed for training)
    desc="Preprocessing audio + text",         # Progress bar label
    load_from_cache_file=False,               # Force recompute — don't use stale cached results
)

print(f"\n📊 Preprocessed dataset size    : {len(preprocessed_dataset)} samples")  # Size check
print(f"📊 Preprocessed dataset columns : {preprocessed_dataset.column_names}")    # Column check

# Spot-check one sample to verify shapes are sensible
sample_check: dict = preprocessed_dataset[0]  # Fetch first preprocessed sample
print(f"\n📋 Sample[0] input_values shape : {np.array(sample_check['input_values']).shape}")  # Waveform frames
print(f"📋 Sample[0] labels             : {sample_check['labels']}")    # Token IDs
print(f"📋 Sample[0] input_length       : {sample_check['input_length']}")  # Frame count
print(f"📋 Sample[0] transcription      : {sample_check['transcription']}")  # Text
print("\n✅ Preprocessing complete.")  # Confirm mapping done

🔄 Mapping prepare_dataset over the 500-sample subset...


Preprocessing audio + text:   0%|          | 0/500 [00:00<?, ? examples/s]


📊 Preprocessed dataset size    : 500 samples
📊 Preprocessed dataset columns : ['transcription', 'input_values', 'labels', 'input_length']

📋 Sample[0] input_values shape : (19216,)
📋 Sample[0] labels             : [39, 59, 47, 0, 64, 20, 56, 47, 61, 30, 52, 34, 61, 34, 49, 41, 52]
📋 Sample[0] input_length       : 19216
📋 Sample[0] transcription      : మోసం చేస్తున్నారు

✅ Preprocessing complete.


## 🧠 Section 7 — Load `facebook/wav2vec2-xls-r-300m`

**Rationale:** XLS-R 300M was pre-trained on 436K hours of multilingual speech including Indian languages, making it a strong starting point for Telugu fine-tuning. We freeze the convolutional feature encoder (the waveform-to-features stem) because its weights are already well-trained for raw audio feature extraction and updating them would require much more data and time.

In [8]:
# ═══════════════════════════════════════════════════════════════
# SECTION 7 — LOAD WAV2VEC2 MODEL
# ═══════════════════════════════════════════════════════════════

MODEL_CHECKPOINT: str = "facebook/wav2vec2-xls-r-300m"  # HuggingFace Hub model ID

print(f"🧠 Loading '{MODEL_CHECKPOINT}'...")  # Progress message

# Load the pretrained model with a custom CTC head sized to our Telugu vocabulary
# ignore_mismatched_sizes=True: the pretrained lm_head has a different vocab size;
#   this flag tells HF to reinitialize the CTC head with our vocab size instead of erroring
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_CHECKPOINT,                               # Download/cache from Hub
    attention_dropout=0.1,                          # Dropout on self-attention weights
    hidden_dropout=0.1,                             # Dropout on hidden state outputs
    feat_proj_dropout=0.0,                          # No dropout on feature projection layer
    mask_time_prob=0.05,                            # SpecAugment: 5% of time steps masked
    mask_time_length=10,                            # Each masked span covers 10 time steps
    ctc_loss_reduction="mean",                      # Average CTC loss over batch (not sum)
    ctc_zero_infinity=True,                         # Replace inf CTC losses with 0 (stability)
    pad_token_id=processor.tokenizer.pad_token_id,  # Tell model which ID is the CTC blank
    vocab_size=tokenizer.vocab_size,                # Size of our Telugu character vocabulary
    ignore_mismatched_sizes=True,                   # Reinitialize LM head with our vocab size
)

# Freeze the convolutional feature encoder (first 7 CNN blocks that process raw waveforms)
# These layers learned general audio features during pretraining on 436K hours;
# freezing them reduces trainable params and prevents catastrophic forgetting
model.freeze_feature_encoder()  # Sets requires_grad=False on all feature_extractor layers

# Move the entire model to GPU memory
model = model.to(DEVICE)  # Transfer all parameters and buffers to CUDA device

# ── Count parameters ──
total_params: int = sum(
    param.numel() for param in model.parameters()  # numel() = number of elements in tensor
)
trainable_params: int = sum(
    param.numel() for param in model.parameters() if param.requires_grad  # Only unfrozen
)
frozen_params: int = total_params - trainable_params  # Frozen = total - trainable

print(f"\n📊 Model Parameter Summary:")
print(f"   Total parameters      : {total_params:>12,}")     # All params
print(f"   Trainable parameters  : {trainable_params:>12,}") # Params that will be updated
print(f"   Frozen parameters     : {frozen_params:>12,}")    # Params frozen (feature encoder)
print(f"   Model device          : {next(model.parameters()).device}")  # Confirm GPU placement

# Report VRAM usage after loading the model
vram_used_gb: float = torch.cuda.memory_allocated(0) / 1e9  # VRAM consumed so far in GB
print(f"   VRAM used (post-load) : {vram_used_gb:.2f} GB")
print("\n✅ Model loaded and moved to GPU.")  # Confirm model ready

🧠 Loading 'facebook/wav2vec2-xls-r-300m'...


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
project_q.weight             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



📊 Model Parameter Summary:
   Total parameters      :  315,507,395
   Trainable parameters  :  311,297,219
   Frozen parameters     :    4,210,176
   Model device          : cuda:0
   VRAM used (post-load) : 1.26 GB

✅ Model loaded and moved to GPU.


## 🔬 Section 8 — Dummy Forward Pass

**Rationale:** Before training, we verify the model can perform a full forward pass end-to-end: input waveforms in → logits out → CTC loss computed. This catches shape mismatches, vocab size errors, and device placement bugs without running a whole training loop. We use `-100` for label padding because `0` is a valid Telugu character ID in our vocabulary.

In [9]:
# ═══════════════════════════════════════════════════════════════
# SECTION 8 — DUMMY FORWARD PASS (2 SAMPLES)
# ═══════════════════════════════════════════════════════════════

FORWARD_BATCH_SIZE: int = 2   # Small batch: just enough to test padding logic
LABEL_PAD_ID: int = -100      # -100 tells CTC loss to ignore these positions (not a real token)

# ── Extract 2 raw waveforms from the preprocessed dataset ──
raw_waveforms: list = [
    np.array(preprocessed_dataset[i]["input_values"])  # Each element: 1D float32 numpy array
    for i in range(FORWARD_BATCH_SIZE)  # Take the first 2 samples
]

raw_labels: list = [
    preprocessed_dataset[i]["labels"]  # Each element: list of integer label IDs
    for i in range(FORWARD_BATCH_SIZE)
]

# ── Pad input_values to the length of the longest waveform in the batch ──
max_waveform_length: int = max(arr.shape[0] for arr in raw_waveforms)  # Longest waveform (frames)
padded_input_values: np.ndarray = np.zeros(
    (FORWARD_BATCH_SIZE, max_waveform_length), dtype=np.float32  # Zero-pad shorter waveforms
)
for batch_idx, waveform in enumerate(raw_waveforms):  # Fill in each waveform
    padded_input_values[batch_idx, :waveform.shape[0]] = waveform  # Left-align, right-pad with 0

# ── Pad labels to the length of the longest label sequence in the batch ──
max_label_length: int = max(len(label_seq) for label_seq in raw_labels)  # Longest label list
padded_labels: np.ndarray = np.full(
    (FORWARD_BATCH_SIZE, max_label_length), fill_value=LABEL_PAD_ID, dtype=np.int64
)  # Fill with -100 so CTC loss ignores padded positions
for batch_idx, label_seq in enumerate(raw_labels):  # Fill in each label sequence
    padded_labels[batch_idx, :len(label_seq)] = label_seq  # Left-align, right-pad with -100

# ── Convert numpy arrays to GPU tensors ──
input_values_tensor: torch.Tensor = torch.tensor(
    padded_input_values, dtype=torch.float32, device=DEVICE  # Move to CUDA
)
labels_tensor: torch.Tensor = torch.tensor(
    padded_labels, dtype=torch.long, device=DEVICE  # Labels must be LongTensor for CTC
)

# ── Run the forward pass ──
model.eval()  # Set to eval mode (disables dropout) for the forward pass test
with torch.no_grad():  # Disable gradient computation — we're just checking shapes and loss
    forward_output = model(
        input_values=input_values_tensor,  # Padded waveform batch: (B, T)
        labels=labels_tensor,              # Padded label batch: (B, L) with -100 for padding
    )

# ── Report shapes and loss ──
print("🔬 Dummy Forward Pass Results:")
print(f"   input_values shape : {input_values_tensor.shape}")    # (B, T) where T=max_waveform_len
print(f"   labels shape       : {labels_tensor.shape}")          # (B, L) where L=max_label_len
print(f"   logits shape       : {forward_output.logits.shape}")  # (B, T', vocab_size)
print(f"   CTC loss value     : {forward_output.loss.item():.4f}")  # Scalar loss

# Assert the last dimension of logits equals our vocabulary size
assert forward_output.logits.shape[-1] == tokenizer.vocab_size, (
    f"❌ Vocab size mismatch: logits last dim {forward_output.logits.shape[-1]} "
    f"!= tokenizer.vocab_size {tokenizer.vocab_size}"
)
print(f"\n✅ logits last dim == vocab_size ({tokenizer.vocab_size}) — vocab wiring confirmed.")

🔬 Dummy Forward Pass Results:
   input_values shape : torch.Size([2, 34208])
   labels shape       : torch.Size([2, 22])
   logits shape       : torch.Size([2, 106, 67])
   CTC loss value     : 19.3241

✅ logits last dim == vocab_size (67) — vocab wiring confirmed.


## 🏋️ Section 9 — Dummy Training Loop (3–5 Steps)

**Rationale:** Running a few manual gradient-update steps confirms that: (1) the optimizer can update the trainable parameters, (2) the CTC loss decreases (or at least changes), and (3) the full decode pipeline (logits → greedy decode → readable text) produces output without crashing. This is the last validation gate before the full fine-tuning run.

In [10]:
# ═══════════════════════════════════════════════════════════════
# SECTION 9 — DUMMY TRAINING LOOP (3–5 STEPS)
# ═══════════════════════════════════════════════════════════════

NUM_DUMMY_STEPS: int = 5      # Number of gradient update steps to run
LEARNING_RATE: float = 3e-5   # AdamW learning rate (standard for Wav2Vec2 fine-tuning)

# Switch model to training mode — enables dropout and SpecAugment masking
model.train()  # Sets all layers to training mode

# AdamW: Adam optimizer with decoupled weight decay, preferred for transformer fine-tuning
# We pass only trainable parameters (frozen encoder params are automatically excluded
# because requires_grad=False prevents their gradients from being stored or updated)
optimizer = torch.optim.AdamW(
    filter(lambda param: param.requires_grad, model.parameters()),  # Trainable params only
    lr=LEARNING_RATE,  # Learning rate for all parameter groups
)

step_loss_history: list = []  # Record loss at each step for the summary
last_step_logits: torch.Tensor = None  # Will hold logits from the final step for decode test

print(f"🏋️  Running {NUM_DUMMY_STEPS} dummy training steps with AdamW (lr={LEARNING_RATE})...\n")

for step_idx in range(NUM_DUMMY_STEPS):  # Iterate each training step

    optimizer.zero_grad()  # Clear gradients from the previous step before computing new ones

    # Forward pass: compute logits and CTC loss together
    # input_values and labels are the same padded tensors built in Section 8
    train_output = model(
        input_values=input_values_tensor,  # Padded waveform batch (B, T) on GPU
        labels=labels_tensor,              # Padded labels (B, L) with -100 padding on GPU
    )

    step_loss: torch.Tensor = train_output.loss  # Scalar CTC loss for this step
    step_loss_history.append(step_loss.item())   # Detach and store as Python float

    step_loss.backward()   # Backpropagate: compute gradients for all trainable parameters
    optimizer.step()       # Apply gradients: update trainable parameter values

    print(f"   Step {step_idx + 1}/{NUM_DUMMY_STEPS} — Loss: {step_loss.item():.4f}")  # Progress log

    last_step_logits = train_output.logits.detach()  # Save logits from the final step

# ── Decode the last step's logits to verify the full decode pipeline ──
print("\n🔊 Decoding last step's logits (greedy CTC decode)...")

# Greedy CTC: take argmax over vocab dimension at each time step
predicted_token_ids: torch.Tensor = torch.argmax(
    last_step_logits, dim=-1  # Shape: (B, T') — one predicted token ID per time step
)

# Decode predicted IDs back to text using the processor
# skip_special_tokens=True removes [PAD] and [UNK] from the output string
decoded_predictions: list = processor.batch_decode(
    predicted_token_ids,         # Tensor of predicted token IDs: (B, T')
    skip_special_tokens=True,    # Remove [PAD] / [UNK] from displayed output
)

print("\n📋 Decoded predictions from the last training step:")
for decode_idx, decoded_text in enumerate(decoded_predictions):  # Print each prediction
    original_text: str = preprocessed_dataset[decode_idx]["transcription"]  # Ground truth
    print(f"   [{decode_idx}] Ground truth : {repr(original_text)}")   # Show reference text
    print(f"   [{decode_idx}] Prediction   : {repr(decoded_text)}")    # Show model output
    print()  # Blank line between samples

print("✅ Dummy training loop complete — pipeline is end-to-end verified.")  # Confirm success

🏋️  Running 5 dummy training steps with AdamW (lr=3e-05)...

   Step 1/5 — Loss: 19.3743
   Step 2/5 — Loss: 19.1421
   Step 3/5 — Loss: 19.1296
   Step 4/5 — Loss: 18.9924
   Step 5/5 — Loss: 18.8540

🔊 Decoding last step's logits (greedy CTC decode)...

📋 Decoded predictions from the last training step:
   [0] Ground truth : 'మోసం చేస్తున్నారు'
   [0] Prediction   : 'ఞాసఞసఃఢఞణభమఛభసణభసభసభఞసఢసమవొఃసశణభబణహవొణథొభ'

   [1] Ground truth : 'మీరు వెంటనే ఒక డబ్బాలో'
   [1] Prediction   : 'వసఎణథభ్ఐఉఢంఐచఢఠఢెపఢంభఢఐఛఢఎణభేూకణకణేవఒ్భసబవభొఠొసభౖభౖంేొేమణ్చఢఈఢఫసభ'

✅ Dummy training loop complete — pipeline is end-to-end verified.


## 📝 Section 10 — Summary

**Rationale:** A consolidated printout of all key metrics from this notebook makes it easy to verify the prototype run at a glance and copy values into experiment logs.

In [11]:
# ═══════════════════════════════════════════════════════════════
# SECTION 10 — SUMMARY
# ═══════════════════════════════════════════════════════════════

print("=" * 60)
print("📝 PROTOTYPE VALIDATION SUMMARY")
print("=" * 60)

# ── Vocabulary & Tokenizer ──
print("\n🔤 Vocabulary & Tokenizer:")
print(f"   Vocab size (Telugu chars + 3 special) : {tokenizer.vocab_size}")  # Total tokens
print(f"   [PAD] token ID                        : {tokenizer.pad_token_id}")  # CTC blank ID
print(f"   [UNK] token ID                        : {tokenizer.unk_token_id}")  # Unknown ID
print(f"   Word delimiter '|' ID                 : {tokenizer.convert_tokens_to_ids('|')}")  # Word sep

# ── Model Parameters ──
print("\n🧠 Model Parameters:")
print(f"   Total parameters      : {total_params:>12,}")      # All model params
print(f"   Trainable parameters  : {trainable_params:>12,}")  # Updated by optimizer
print(f"   Frozen parameters     : {frozen_params:>12,}")     # Feature encoder (frozen)
print(f"   Trainable fraction    : {trainable_params/total_params*100:.1f}%")  # % of params updated

# ── Tensor Shapes ──
print("\n📐 Tensor Shapes (batch_size=2):")
print(f"   input_values  : {tuple(input_values_tensor.shape)}")    # (B, T)
print(f"   labels        : {tuple(labels_tensor.shape)}")          # (B, L)
print(f"   logits        : {tuple(last_step_logits.shape)}")       # (B, T', vocab)

# ── Loss Values ──
print("\n📉 CTC Loss per Dummy Training Step:")
for step_num, loss_val in enumerate(step_loss_history, start=1):  # Print each step's loss
    print(f"   Step {step_num}: {loss_val:.4f}")  # Step number and loss value

# ── Output Artifacts ──
print("\n📦 Output Artifacts:")
print(f"   vocab.json                    : {VOCAB_SAVE_PATH}")    # Character vocabulary
print(f"   telugu_wav2vec2_processor/    : {PROCESSOR_SAVE_PATH}")  # Tokenizer + extractor

print("\n" + "=" * 60)
print("✅ All sections passed. Pipeline is ready for full fine-tuning in File 3.")
print("=" * 60)

📝 PROTOTYPE VALIDATION SUMMARY

🔤 Vocabulary & Tokenizer:
   Vocab size (Telugu chars + 3 special) : 67
   [PAD] token ID                        : 66
   [UNK] token ID                        : 65
   Word delimiter '|' ID                 : 64

🧠 Model Parameters:
   Total parameters      :  315,507,395
   Trainable parameters  :  311,297,219
   Frozen parameters     :    4,210,176
   Trainable fraction    : 98.7%

📐 Tensor Shapes (batch_size=2):
   input_values  : (2, 34208)
   labels        : (2, 22)
   logits        : (2, 106, 67)

📉 CTC Loss per Dummy Training Step:
   Step 1: 19.3743
   Step 2: 19.1421
   Step 3: 19.1296
   Step 4: 18.9924
   Step 5: 18.8540

📦 Output Artifacts:
   vocab.json                    : ./vocab.json
   telugu_wav2vec2_processor/    : ./telugu_wav2vec2_processor

✅ All sections passed. Pipeline is ready for full fine-tuning in File 3.


## ➡️ Next Steps — File 3

**`03_evaluation_and_inference.ipynb`** will:
1. Load `./telugu_wav2vec2_processor/` with `Wav2Vec2Processor.from_pretrained()`
2. Preprocess the **full** `./telugu_asr_clean_dataset/` with caching enabled
3. Run a full HuggingFace `Trainer` fine-tuning job with:
   - `DataCollatorCTCWithPadding` for dynamic batch padding
   - `WERMetric` callback for per-epoch WER tracking
   - `fp16=True` for mixed-precision training on the A6000
4. Evaluate on the held-out test split (WER, CER)
5. Run live inference demos with greedy and beam-search decoding